In [10]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
import time
from datetime import datetime, timedelta
import pandas as pd
import urllib.parse
import random
import json
import os

# 봇 탐지를 피하기 위해 user agent로 설정
user_agents = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:138.0) Gecko/20100101 Firefox/138.0',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36',
]

options = Options()
options.add_argument(f'user-agent={random.choice(user_agents)}')
options.add_experimental_option('excludeSwitches', ['enable-automation'])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument('--disable-blink-features=AutomationControlled')

# Chrome 드라이버 자동 다운로드 및 설정
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# navigator.webdriver 플래그 제거
driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
    'source': 'Object.defineProperty(navigator, "webdriver", {get: () => undefined})'
})

# 변수 정의 (날짜 형식: 'YYYY.MM.DD')
start_date = '2025.11.01'
end_date = '2025.11.30'
query = 'KT'

# 임시 링크 파일 경로
temp_links_path = os.path.join(os.getcwd(), f'{query}_{start_date}_{end_date}_temp_links.json')

# 창 열기는 아래 셀의 날짜 루프 안에서 날짜별로 자동 실행됩니다

In [ ]:
## 날짜를 하루씩 쪼개서 링크 수집 후 임시 파일에 누적 저장
## 이미 임시 파일이 있으면 이어서 누적 (중복은 자동 제거)

# 임시 파일에 기존 링크가 있으면 불러오기
if os.path.exists(temp_links_path):
    with open(temp_links_path, 'r', encoding='utf-8') as f:
        checkpoint = json.load(f)
    all_links_set = set(checkpoint['links'])
    last_collected_date = checkpoint['last_date']  # 마지막으로 완료한 날짜
    print(f"기존 임시 파일에서 링크 {len(all_links_set)}개 불러옴 — {last_collected_date} 다음부터 이어서 수집")
else:
    all_links_set = set()
    last_collected_date = None
    print("새로 링크 수집 시작")

# start_date ~ end_date를 하루씩 순회
current = datetime.strptime(start_date, '%Y.%m.%d')
end = datetime.strptime(end_date, '%Y.%m.%d')

while current <= end:
    day_str = current.strftime('%Y.%m.%d')  # 'YYYY.MM.DD' 형식

    # 이미 수집 완료한 날짜면 건너뜀
    if last_collected_date and day_str <= last_collected_date:
        print(f"{day_str} — 이미 수집 완료, 건너뜀")
        current += timedelta(days=1)
        continue

    # 하루짜리 검색 URL 생성
    encoded_query = urllib.parse.quote(query)
    # 주소 원문 해당 구간 : so%3Ar%2Cp%3Afrom20220816to20220831 -> ':', ',' 처리를 위해 인코딩 필요
    nso_value = f"so:r,p:from{day_str.replace('.', '')}to{day_str.replace('.', '')}"
    url = f'https://search.naver.com/search.naver?ssc=tab.news.all&query={encoded_query}&sm=tab_opt&sort=2&photo=0&field=0&pd=3&ds={day_str}&de={day_str}&docid=&related=0&mynews=0&office_type=0&office_section_code=0&news_office_checked=&nso={urllib.parse.quote(nso_value)}&is_sug_officeid=0&office_category=&service_area=2'

    driver.get(url)

    # 페이지 끝까지 내리기
    # https://m.blog.naver.com/lmj4160/222462966573 에서 코드 발췌
    last_height = driver.execute_script("return document.body.scrollHeight")
    PAUSE_SEC = 1.5  # 한 번에 끝까지 내리고 싶으면 1초 이상으로 설정하는 것을 권장

    while True:
        # 스크롤 끝까지 내리기
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        # 스크롤 내린 후 페이지 로딩을 위한 시간이 필요하다면, PAUSE_SEC에 숫자 입력
        time.sleep(PAUSE_SEC)

        # 스크롤 내린 후의 페이지 높이 = new_height
        new_height = driver.execute_script("return document.body.scrollHeight")

        # 더이상 스크롤이 내려가지 않으면 스크롤 내리는 반복문 멈춤 (=last_height와 new_height 같으면 멈춤)
        if new_height == last_height:
            # 네트워크 지연으로 콘텐츠가 늦게 로딩될 수 있으므로 한 번 더 대기 후 재확인
            time.sleep(2.0)
            final_height = driver.execute_script("return document.body.scrollHeight")
            if final_height == new_height:
                break  # 재확인에도 높이 변화 없으면 진짜 끝
            else:
                last_height = final_height  # 추가 콘텐츠가 로딩됐으면 계속 스크롤
                continue

        # 스크롤 내린 후의 페이지 높이(new_height)를 현재 페이지 높이(last_height) 변수에 저장
        last_height = new_height

    # 링크 수집 후 set에 추가 (중복 자동 제거)
    a_tags = driver.find_elements(By.XPATH, '//a[contains(@href, "n.news.naver.com")]')
    day_links = {a.get_attribute('href') for a in a_tags}
    before = len(all_links_set)
    all_links_set.update(day_links)
    added = len(all_links_set) - before

    print(f"{day_str} — {len(day_links)}건 수집 / 신규 {added}건 추가 / 누적 {len(all_links_set)}건")

    # 하루치 수집 후 임시 파일에 즉시 저장 (중간에 끊겨도 누적 보존 + 마지막 완료 날짜 기록)
    with open(temp_links_path, 'w', encoding='utf-8') as f:
        json.dump({'links': list(all_links_set), 'last_date': day_str}, f, ensure_ascii=False)

    current += timedelta(days=1)

# 월별 링크 최종 저장본 생성
naver_news_links = list(all_links_set)

save_dir = os.path.join(os.getcwd(), 'data')
os.makedirs(save_dir, exist_ok=True)
links_file_name = f"링크_{query}_{start_date.replace('.', '')[2:]}_{end_date.replace('.', '')[2:]}.json"
links_save_path = os.path.join(save_dir, links_file_name)

with open(links_save_path, 'w', encoding='utf-8') as f:
    json.dump(naver_news_links, f, ensure_ascii=False)

# 임시 파일 삭제
if os.path.exists(temp_links_path):
    os.remove(temp_links_path)

print(f"\n링크 수집 완료 — 총 {len(naver_news_links)}개")
print(f"월별 저장본 저장 완료: {links_save_path}")

In [ ]:
# 브라우저 창 닫기
driver.quit()